# Round 3 research

Single notebook covering EDA, vol-surface analysis, and backtest review.
Modules: `options.py` (BS / IV / TTE), `backtest.py` (data loaders + engine).

Sections:
1. Load round-3 prices and trades
2. EDA: mids, spreads, depth
3. Implied volatility: smile per day, surface, greeks
4. Backtest review: equity curve, per-product PnL, trade overlay

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import options
import backtest as bt

## 1. Load round-3 data

In [ ]:
DAYS = (0, 1, 2)
prices = bt.load_prices_df(days=DAYS)
trades = bt.load_trades_df(days=DAYS)
print(f'prices: {len(prices):,}  trades: {len(trades):,}')
prices.groupby('product').agg(
    n=('mid_price', 'size'),
    mid_mean=('mid_price', 'mean'),
    mid_min=('mid_price', 'min'),
    mid_max=('mid_price', 'max'),
).round(2)

## 2. EDA

In [ ]:
def gts(df):
    return df['day'].astype(int) * bt.TS_PER_DAY + df['timestamp'].astype(int)

def plot_mid(prods, ax=None):
    if ax is None: fig, ax = plt.subplots(figsize=(12, 4))
    for p in prods:
        sub = prices[prices['product'] == p]
        if sub.empty: continue
        ax.plot(gts(sub), sub['mid_price'], label=p, linewidth=1)
    ax.set_xlabel('global ts'); ax.set_ylabel('mid'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    return ax

plot_mid(['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT']);
plt.title('Commodity mid prices'); plt.show()
plot_mid([f'VEV_{k}' for k in (4500, 5000, 5300, 5500)]);
plt.title('VEV mid prices (selected strikes)'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for p in ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT']:
    sub = prices[prices['product'] == p].dropna(subset=['bid_price_1', 'ask_price_1'])
    ax.plot(gts(sub), sub['ask_price_1'] - sub['bid_price_1'], label=p, linewidth=1)
ax.set_xlabel('global ts'); ax.set_ylabel('spread'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Bid-ask spread'); plt.show()

In [ ]:
# Depth scatter for VELVETFRUIT_EXTRACT, day 0
sub = prices[(prices['product'] == 'VELVETFRUIT_EXTRACT') & (prices['day'] == 0)].copy()
sub['gts'] = gts(sub)
fig, ax = plt.subplots(figsize=(14, 5))
for lvl, alpha in zip((1, 2, 3), (0.5, 0.3, 0.2)):
    bp, bv = sub[f'bid_price_{lvl}'], sub[f'bid_volume_{lvl}']
    ap, av = sub[f'ask_price_{lvl}'], sub[f'ask_volume_{lvl}']
    m = bp.notna() & bv.notna()
    ax.scatter(sub.loc[m, 'gts'], bp[m], s=bv[m]*1.5, c='steelblue', alpha=alpha, label=f'bid L{lvl}' if lvl==1 else None)
    m = ap.notna() & av.notna()
    ax.scatter(sub.loc[m, 'gts'], ap[m], s=av[m]*1.5, c='crimson', alpha=alpha, label=f'ask L{lvl}' if lvl==1 else None)
ax.plot(sub['gts'], sub['mid_price'], color='black', linewidth=0.7, label='mid')
ax.set_title('VELVETFRUIT_EXTRACT depth (day 0)'); ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.show()

## 3. Implied volatility

In [ ]:
spot = prices[prices['product'] == bt.VEV_UNDERLYING][['day', 'timestamp', 'mid_price']].rename(columns={'mid_price': 'spot'})
vev = prices[prices['product'].str.startswith('VEV_')].copy()
vev['strike'] = vev['product'].str.replace('VEV_', '').astype(float)
vev = vev.merge(spot, on=['day', 'timestamp']).dropna(subset=['mid_price'])
vev['tte'] = [options.tte_days(d, t) for d, t in zip(vev['day'], vev['timestamp'])]
vev['iv'] = [options.implied_vol(p, S, K, T) for p, S, K, T in zip(vev['mid_price'], vev['spot'], vev['strike'], vev['tte'])]
vev['log_moneyness'] = np.log(vev['strike'] / vev['spot'])
print(f'IVs computed: {vev["iv"].notna().sum():,} / {len(vev):,}')
vev['iv'].describe().round(5)

In [ ]:
# Smile per day at a few snapshots
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, day in zip(axes, sorted(vev['day'].unique())):
    slc = vev[(vev['day'] == day) & vev['iv'].notna()]
    if slc.empty:
        ax.set_title(f'day {day}: no IVs'); continue
    sample_ts = sorted(slc['timestamp'].unique())[::max(len(slc['timestamp'].unique())//4, 1)]
    for ts in sample_ts[:5]:
        s = slc[slc['timestamp'] == ts].sort_values('strike')
        ax.plot(s['strike'], s['iv'], marker='o', label=f'ts={ts}', alpha=0.7)
    ax.set_xlabel('strike'); ax.set_title(f'day {day}'); ax.grid(alpha=0.3); ax.legend(fontsize=7)
axes[0].set_ylabel('IV (per √day)'); fig.suptitle('Vol smile'); plt.tight_layout(); plt.show()

In [ ]:
# 3D surface (sub-sampled)
sub = vev.dropna(subset=['iv']).iloc[::2000]
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(sub['tte'], sub['strike'], sub['iv'], c=sub['iv'], cmap='viridis', s=8, alpha=0.6)
ax.set_xlabel('TTE (days)'); ax.set_ylabel('strike'); ax.set_zlabel('IV')
fig.colorbar(sc, shrink=0.6); ax.set_title('Implied vol surface'); plt.show()

In [ ]:
# Greeks for a representative ATM call
S = np.linspace(5000, 5500, 200); K, T, sigma = 5300, 5.0, 0.05
d = [options.delta(s, K, T, sigma) for s in S]
g = [options.gamma(s, K, T, sigma) for s in S]
v = [options.vega(s, K, T, sigma) for s in S]
fig, ax = plt.subplots(figsize=(10, 4)); ax2 = ax.twinx()
ax.plot(S, d, label='delta', c='C0')
ax.plot(S, np.array(g) * 100, label='gamma×100', c='C1')
ax2.plot(S, v, label='vega', c='C2', linestyle='--')
ax.axvline(K, c='grey', linewidth=0.7, linestyle=':')
ax.set_xlabel('spot'); ax.set_ylabel('delta, gamma×100'); ax2.set_ylabel('vega')
ax.legend(loc='upper left', fontsize=8); ax2.legend(loc='upper right', fontsize=8)
ax.set_title(f'Greeks: K={K} T={T} σ={sigma}'); plt.show()

## 4. Backtest review

Run `python backtest.py --days 0,1,2` from the project root, or call
`bt.run_backtest(...)` here. Loads the most recent JSON in `results/`.

In [ ]:
from trader import Trader
result = bt.run_backtest(Trader(), days=DAYS, matcher='depth')
print(result.summary)

In [ ]:
marks = result.marks_df()
trades_df = result.trades_df()
if marks.empty:
    print('no marks')
else:
    marks['gts'] = gts(marks)
    eq = marks.groupby('gts')['mtm_pnl'].sum().sort_index()
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(eq.index, eq.values, color='black'); ax.fill_between(eq.index, eq.values, alpha=0.2)
    ax.axhline(0, color='grey', linewidth=0.6); ax.set_title('Equity curve (mark-to-mid)')
    ax.set_xlabel('global ts'); ax.set_ylabel('PnL'); ax.grid(alpha=0.3); plt.show()

    pivot = marks.pivot_table(index='gts', columns='product', values='mtm_pnl', aggfunc='last').ffill().fillna(0.0)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.stackplot(pivot.index, pivot.T, labels=pivot.columns, alpha=0.7)
    ax.axhline(0, color='grey', linewidth=0.6); ax.legend(fontsize=7, ncol=3); ax.grid(alpha=0.3)
    ax.set_title('Per-product PnL'); plt.show()

In [ ]:
# Mid + trade-scatter overlay for any product the trader actually trades
if trades_df.empty:
    print('Strategy stubs traded nothing — develop alpha in trader.py and re-run.')
else:
    for product in sorted(trades_df['product'].unique()):
        m = marks[marks['product'] == product]
        t = trades_df[trades_df['product'] == product]
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(m['gts'], m['mid'], color='black', linewidth=0.8)
        buys = t[t['side'] == 'BUY']; sells = t[t['side'] == 'SELL']
        ax.scatter(buys['day']*bt.TS_PER_DAY + buys['timestamp'], buys['price'], marker='^', color='green', s=buys['qty']*8, alpha=0.6)
        ax.scatter(sells['day']*bt.TS_PER_DAY + sells['timestamp'], sells['price'], marker='v', color='red', s=sells['qty']*8, alpha=0.6)
        ax.set_title(product); ax.grid(alpha=0.3); plt.show()